# Eval Correction Loop

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/evaluation/eval-correction-loop). Run the cells in order in a fresh Python 3.12 kernel.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "agent-learning-kit==0.3.7"], check=True);

## Set your credentials

Set the named environment variables or Colab secrets before running, or enter them at the hidden prompts. Values are not displayed. For self-hosting, set `FI_BASE_URL` to your platform API.

In [ ]:
import os
from getpass import getpass


def require_secret(name):
    if os.environ.get(name):
        return
    value = None
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
    if not value:
        value = getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value


for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    require_secret(name)

## Score the batch with a built-in eval

In [ ]:
import json
import os
from pathlib import Path
from uuid import uuid4
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

# Synthetic support replies. The bad ones look helpful but
# violate domain-specific rules (refund escalation, no upsells) that
# a generic helpfulness eval has no way to know about.
samples = [
    {
        "id": "r1",
        "user_query": "I forgot my password and can't log in.",
        "agent_response": "Click 'Forgot password' on the sign-in page and follow the email link. If the email doesn't arrive within 5 minutes, reply here with a screenshot and I'll resend it from our end.",
        "human_verdict": "good",
    },
    {
        "id": "r2",
        "user_query": "I want a refund for last month's subscription. The service was down 3 times.",
        "agent_response": "I'm sorry to hear about the disruption. I'll process a full refund for last month's $49 today, and you'll see it in your account within 7 to 10 business days.",
        "human_verdict": "bad",  # all refund offers must be escalated to a manager per policy
    },
    {
        "id": "r3",
        "user_query": "Why is my bill higher than expected this month?",
        "agent_response": "Your Pro plan has overage fees for usage above 10GB. You used 15GB this billing cycle, which added $12 to your bill. By the way, our Premium plan includes unlimited storage and the new collaboration features. Want me to upgrade you?",
        "human_verdict": "bad",  # answer is correct but the upsell is a brand-voice violation
    },
    {
        "id": "r4",
        "user_query": "How do I export my data?",
        "agent_response": "Go to Settings > Data > Export. You can pick CSV or JSON, and we'll email a download link within 30 minutes.",
        "human_verdict": "good",
    },
]

baseline_results = []
for s in samples:
    r = evaluator.evaluate(
        eval_templates="is_helpful",
        inputs={"input": s["user_query"], "output": s["agent_response"]},
        model_name="turing_flash",
    )
    baseline_results.append({
        "id": s["id"],
        "eval_id": r.eval_results[0].eval_id,
        "eval_score": r.eval_results[0].output,
        "eval_reason": r.eval_results[0].reason,
        "human_verdict": s["human_verdict"],
    })

for row in baseline_results:
    print(f"{row['id']}: eval={row['eval_score']!s:>5} | human={row['human_verdict']:>4} | {row['eval_reason'][:80]}")

## Identify the disagreements

In [ ]:
def passed(score):
    verdict = str(score).strip().lower()
    if verdict not in {"passed", "failed", "true", "false"}:
        raise ValueError(f"Expected a pass/fail result, got {score!r}")
    return verdict in {"passed", "true"}

disagreements = [
    r for r in baseline_results
    if passed(r["eval_score"]) != (r["human_verdict"] == "good")
]

print(f"{len(disagreements)} / {len(baseline_results)} disagreed with humans")
for r in disagreements:
    print(f"  {r['id']}: eval said {r['eval_score']}, human said {r['human_verdict']}")
    print(f"    reason: {r['eval_reason'][:120]}")

## Encode the corrections as a custom eval

In [ ]:
from fi.evals import EvalTemplateManager

rule_prompt = """\
You evaluate customer-support replies for a SaaS product.

A reply passes ONLY if ALL of the following hold:
1. Stays focused on the user's specific issue. No marketing language, no upsells, no pivots to other products.
2. Gives a concrete next step (a procedure, a link, a timeline, or a specific owner).
3. Does NOT commit to a refund, credit, or policy exception. Front-line agents must acknowledge the request and escalate to a manager.
4. Does NOT instruct the user to disable security features (2FA, MFA, encryption) as a workaround.

Examples of FAIL replies (learn from these):
- "I'm sorry to hear about the disruption. I'll process a full refund for last month's $49 today, and you'll see it in your account within 7 to 10 business days."
  -> FAIL: rule 3. Front-line agents can't commit to refunds. Should acknowledge and escalate.
- "Your Pro plan has overage fees for usage above 10GB. You used 15GB this billing cycle, which added $12 to your bill. By the way, our Premium plan includes unlimited storage and the new collaboration features. Want me to upgrade you?"
  -> FAIL: rule 1. Pivots from billing question to a sales pitch.

Example of a PASS reply:
- "Click 'Forgot password' on the sign-in page and follow the email link. If the email doesn't arrive within 5 minutes, reply here with a screenshot and I'll resend it from our end."
  -> PASS: focused on the issue, concrete next step, clear escalation path.

Now evaluate this reply.

User query: {{user_query}}
Agent response: {{agent_response}}
"""

template_manager = EvalTemplateManager(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

response = template_manager.create_template(
    name=f"support_reply_quality_v1_{uuid4().hex[:8]}",
    instructions=rule_prompt,
    model="turing_flash",
    output_type="pass_fail",
)
print(response)
# `name` is what you pass to `evaluator.evaluate(eval_templates=...)` in the next step.
# The required keys (user_query, agent_response) are inferred from the {{...}}
# placeholders in `instructions`, you don't declare them separately.

## Re-score the same batch and measure agreement

In [ ]:
calibrated_results = []
for s in samples:
    r = evaluator.evaluate(
        eval_templates=response.name,
        model_name="turing_flash",
        inputs={"user_query": s["user_query"], "agent_response": s["agent_response"]},
    )
    calibrated_results.append({
        "id": s["id"],
        "eval_id": r.eval_results[0].eval_id,
        "eval_score": r.eval_results[0].output,
        "human_verdict": s["human_verdict"],
    })

agreement = sum(
    1 for r in calibrated_results
    if passed(r["eval_score"]) == (r["human_verdict"] == "good")
)
Path("support-eval-results.json").write_text(json.dumps({
    "template": {"id": response.id, "name": response.name},
    "baseline": baseline_results, "calibrated": calibrated_results,
}, indent=2))
print(f"agreement: {agreement} / {len(samples)} ({100 * agreement / len(samples):.0f}%)")
for r in calibrated_results:
    match = "OK" if passed(r["eval_score"]) == (r["human_verdict"] == "good") else "MISS"
    print(f"  {match} {r['id']}: eval={r['eval_score']} human={r['human_verdict']}")

## Iterate when agreement plateaus below your bar

In [ ]:
# After collecting fresh disagreements...
rule_prompt_v2 = rule_prompt + """

Additional FAIL example (learn from this):
- "Try disabling 2FA temporarily so you can log in, then re-enable it once you're past the issue."
  -> FAIL: rule 4. Never instruct users to disable security features. Offer a recovery code or escalate to security ops.
"""

# Re-register with template_manager.create_template(name="support_reply_quality_v2", ...)
# and compare scores side-by-side.

Read the agreement count alongside the baseline disagreements. The rubric quotes some of these replies, so use fresh examples before judging how well it generalizes.